# Notebook 25 IsoBlur DeLong

**Iso-blur paired DeLong test, RAD-DINO vs DINOv3.**

Compute paired DeLong p-value for RAD-DINO vs DINOv3 on the iso-blur detection
task (clean-vs-perturbed binary classification).

Reuses cached CLS embeddings from exp03 — no model forward passes required.

Usage (NIH):
    DATASET=nih python compute_iso_delong.py


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory, vindr (where applicable)
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None when running gated models locally


In [ ]:
# Apply papermill parameters to environment
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DATASET = os.environ.get("DATASET", "nih")
WORK = Path(os.environ.get(
    "OUTPUTS_DIR", "/home/saptpurk/embeddings-noise-eliminators/outputs"))
EXP03_DIR = WORK / f"v4_exp03_iso_motion_blur_{DATASET}"
OUT = WORK / "v4_iso_delong"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
C_GRID = (0.01, 0.1, 1.0, 10.0)
PATCHES = (4, 8)
MODELS = ("raddino", "dinov3")


def _midrank(x):
    order = np.argsort(x, kind="mergesort")
    n = len(x)
    ranks = np.empty(n, dtype=float)
    x_sorted = x[order]
    i = 0
    while i < n:
        j = i
        while j < n and x_sorted[j] == x_sorted[i]:
            j += 1
        ranks[order[i:j]] = (i + j + 1) / 2.0
        i = j
    return ranks


def _auc_components(y_true, y_score):
    """Return (auc, V10, V01) per Sun & Xu / DeLong."""
    pos = y_true == 1
    neg = y_true == 0
    m = int(pos.sum())
    n = int(neg.sum())
    sp = y_score[pos]
    sn = y_score[neg]
    TX = _midrank(sp)
    TY = _midrank(sn)
    TZ = _midrank(np.concatenate([sp, sn]))
    Tp = TZ[:m]
    Tn = TZ[m:]
    V10 = (Tp - TX) / n
    V01 = 1.0 - (Tn - TY) / m
    return float(V10.mean()), V10, V01


def delong_paired(y_true, score_a, score_b):
    auc_a, V10_a, V01_a = _auc_components(y_true, score_a)
    auc_b, V10_b, V01_b = _auc_components(y_true, score_b)
    s10 = np.cov(V10_a, V10_b, ddof=1)
    s01 = np.cov(V01_a, V01_b, ddof=1)
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    var = ((s10[0, 0] - 2 * s10[0, 1] + s10[1, 1]) / n_pos
           + (s01[0, 0] - 2 * s01[0, 1] + s01[1, 1]) / n_neg)
    if var <= 0:
        return auc_a, auc_b, float("nan"), float("nan")
    z = (auc_a - auc_b) / np.sqrt(var)
    p = 2.0 * (1.0 - norm.cdf(abs(z)))
    return auc_a, auc_b, float(z), float(p)


def fit_and_score(X_train, y_train, X_test):
    """Match exp03 train_probe: StandardScaler + CV-chosen C logistic."""
    scaler = StandardScaler().fit(X_train)
    Xtr = scaler.transform(X_train)
    Xte = scaler.transform(X_test)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    best_C, best_cv = None, -np.inf
    for C in C_GRID:
        cv_aucs = []
        for tr_idx, va_idx in skf.split(Xtr, y_train):
            clf = LogisticRegression(C=C, max_iter=2000,
                                     class_weight="balanced",
                                     solver="lbfgs", random_state=SEED)
            clf.fit(Xtr[tr_idx], y_train[tr_idx])
            cv_aucs.append(roc_auc_score(
                y_train[va_idx],
                clf.predict_proba(Xtr[va_idx])[:, 1]))
        m = float(np.mean(cv_aucs))
        if m > best_cv:
            best_cv = m
            best_C = C
    clf = LogisticRegression(C=best_C, max_iter=2000,
                             class_weight="balanced",
                             solver="lbfgs", random_state=SEED)
    clf.fit(Xtr, y_train)
    return clf.predict_proba(Xte)[:, 1], best_C


def stack_iso_test_set(model, patches=PATCHES):
    """Concatenate clean and perturbed iso-blur test embeddings across patches.

    Returns (X_train, y_train, X_test, y_test) for the binary detection task.
    Pooling p4 + p8 gives a single test set the manuscript can refer to.
    """
    Xtr_parts, ytr_parts = [], []
    Xte_parts, yte_parts = [], []
    for ps in patches:
        for tag, X_acc, y_acc in (("train", Xtr_parts, ytr_parts),
                                  ("test", Xte_parts, yte_parts)):
            d = np.load(EXP03_DIR / f"emb_{model}_p{ps}_{tag}.npz")
            cln, prt = d["clean"], d["pert"]
            X_acc.append(np.vstack([cln, prt]))
            y_acc.append(np.concatenate([
                np.zeros(len(cln)), np.ones(len(prt))]))
    return (np.vstack(Xtr_parts), np.concatenate(ytr_parts).astype(int),
            np.vstack(Xte_parts), np.concatenate(yte_parts).astype(int))


def run_single_patch(model, ps):
    """Probe trained on one patch size only — used for per-condition reporting."""
    d_tr = np.load(EXP03_DIR / f"emb_{model}_p{ps}_train.npz")
    d_te = np.load(EXP03_DIR / f"emb_{model}_p{ps}_test.npz")
    Xtr = np.vstack([d_tr["clean"], d_tr["pert"]])
    ytr = np.concatenate([np.zeros(len(d_tr["clean"])),
                          np.ones(len(d_tr["pert"]))]).astype(int)
    Xte = np.vstack([d_te["clean"], d_te["pert"]])
    yte = np.concatenate([np.zeros(len(d_te["clean"])),
                          np.ones(len(d_te["pert"]))]).astype(int)
    proba, C = fit_and_score(Xtr, ytr, Xte)
    return yte, proba, C


def main():
    print(f"Dataset: {DATASET}")
    print(f"exp03 dir: {EXP03_DIR}")

    # --- Pooled p4+p8 test set (the headline single p-value) ---
    Xtr_r, ytr_r, Xte_r, yte_r = stack_iso_test_set("raddino")
    Xtr_d, ytr_d, Xte_d, yte_d = stack_iso_test_set("dinov3")
    assert np.array_equal(yte_r, yte_d), "test labels diverge between models"
    proba_r, C_r = fit_and_score(Xtr_r, ytr_r, Xte_r)
    proba_d, C_d = fit_and_score(Xtr_d, ytr_d, Xte_d)
    auc_r, auc_d, z, p = delong_paired(yte_r, proba_r, proba_d)
    print(f"\nPooled p4+p8 test set ({len(yte_r)} samples, "
          f"{int(yte_r.sum())} pos):")
    print(f"  RAD-DINO AUC = {auc_r:.4f}  (best_C={C_r})")
    print(f"  DINOv3   AUC = {auc_d:.4f}  (best_C={C_d})")
    print(f"  paired DeLong z = {z:.3f}  p = {p:.4g}")

    rows = [{
        "dataset": DATASET, "patches": "p4+p8", "n_test": int(len(yte_r)),
        "auc_raddino": auc_r, "auc_dinov3": auc_d,
        "delong_z": z, "delong_p": p,
        "best_C_raddino": float(C_r), "best_C_dinov3": float(C_d),
    }]

    # --- Per-patch breakdown for reference ---
    for ps in PATCHES:
        yte_r, proba_r, C_r = run_single_patch("raddino", ps)
        yte_d, proba_d, C_d = run_single_patch("dinov3", ps)
        assert np.array_equal(yte_r, yte_d)
        auc_r, auc_d, z, p = delong_paired(yte_r, proba_r, proba_d)
        print(f"\np{ps} only ({len(yte_r)} samples):")
        print(f"  RAD-DINO AUC = {auc_r:.4f}   "
              f"DINOv3 AUC = {auc_d:.4f}   z = {z:.3f}   p = {p:.4g}")
        rows.append({
            "dataset": DATASET, "patches": f"p{ps}",
            "n_test": int(len(yte_r)),
            "auc_raddino": auc_r, "auc_dinov3": auc_d,
            "delong_z": z, "delong_p": p,
            "best_C_raddino": float(C_r), "best_C_dinov3": float(C_d),
        })

    df = pd.DataFrame(rows)
    out_p = OUT / f"iso_delong_{DATASET}.parquet"
    df.to_parquet(out_p, index=False)
    print(f"\nWrote {out_p}")

    # Headline p-value: pooled p4+p8 (matches the manuscript phrasing
    # "RAD-DINO vs DINOv3 on the same test set" for the iso-blur figure).
    headline = df[df["patches"] == "p4+p8"].iloc[0]
    summary = {
        f"exp_iso_delong_{DATASET}_p": f"{headline['delong_p']:.3g}",
        f"exp_iso_delong_{DATASET}_z": f"{headline['delong_z']:.2f}",
        f"exp_iso_delong_{DATASET}_auc_raddino": f"{headline['auc_raddino']:.4f}",
        f"exp_iso_delong_{DATASET}_auc_dinov3": f"{headline['auc_dinov3']:.4f}",
    }
    (OUT / f"iso_delong_{DATASET}.json").write_text(json.dumps(summary, indent=2))
    print(f"Headline tokens: {summary}")


# === Main entry-point (papermill will execute) ===
    main()
